# **Progetto Data Mining - Predizione sull'abbandono universitario o rendimento studenti**
#### **Studente**: Rettura Damian Giuseppe
#### **Matricola**: 255457

---

## Descrizione del Progetto

L'obiettivo di questo progetto è sviluppare un modello di classificazione in grado di predire se uno studente universitario completerà gli studi (Graduate) o abbandonerà (Dropout). 
Il dataset utilizzato è estratto da Kaggle ed è focalizzato su metriche demografiche, macroeconomiche e accademiche.
Poiché l'esito per gli studenti attualmente iscritti ("Enrolled") è incerto, il l problema è stato formulato come un task di **classificazione binaria** escludendo questa categoria.

Il progetto segue un workflow strutturato in più fasi:
1. Lettura dati
2. Esplorazione dati
3. Pre-processing e preparazione
4. Applicazione di modelli di classificazione
5. Valutazione e confronto modelli (con focus su metriche avanzate)
6. Considerazioni finali
7. Prospettive di miglioramento


---


# Fase 1: Lettura dei dati

In questa fase viene caricato il dataset sugli studenti.
Viene eseguito un primo controllo esplorativo sulle dimensione della base di dati e sulla presenza di eventuali valori mancanti (null) che potrebbero compromettere l'addestramento.

In [ ]:
import pandas as pd

df = pd.read_csv('dataset.csv')

# dimensioni del dataset iniziale
print(f"Numero di record nel dataset: {df.shape[0]}")
print(f"Numero di colonne (feature + target): {df.shape[1]}")

# controllo valori mancanti
print("\nValori nulli totali nel dataset:", df.isnull().sum().sum())

# visualizzazione delle prime 5 righe
df.head()

# Fase 2: Esplorazione dei dati

L'analisi esplorativa (EDA) ci permette di comprendere  la distribuzione della nostra variabile target.
In questa fase si verifica l'eventuale sbilanciamento tra le classi prima di procedere con la pulizia.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# distribuzione iniziale delle classi nella colonna Target
plt.figure(figsize=(8,4))

# grafico a barre (countplot) per visualizzare quanti studenti ci sono per ogni classe.
# I parametri 'hue' e 'legend=False' servono per evitare warning
sns.countplot(data=df, x='Target',hue='Target', palette='viridis', legend=False)

plt.title('Distribuzione iniziale delle classi (Target)')
plt.xlabel('Stato dello Studente')
plt.ylabel('Numero di studenti')
plt.grid(axis='y', linestyle='--', alpha=0.7)   # griglia orizzontale di riferimento
plt.show()

# valori presenti nella tabella target
print("\nConteggio esatto per classe:")
print(df['Target'].value_counts())

# Fase 3: Pre-Processing e Preparazione

Per ottimizzare le performance dei modelli vado ad effettuare le seguenti operazioni di pulizia:
* Rimozione dei duplicati e dei valori nulli.
* **Filtro del Target:** Rimozione della classe 'Enrolled' per trasformare il problema in classificazione binaria.
* **Mappatura:** Conversione del target testuale in numerico ('Dropout' = 1, 'Graduate' = 0). Il Dropout è la classe positiva che voglio intercettare.
* **Standardizzazione:** Scalatura delle feature numeriche per facilitare l'addestramento di modelli basati su distanze o gradienti.


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

#1. Rimozione dei valori nulli e duplicati
df_clean = df.dropna().drop_duplicates()

#2. Trasformazione in classificazione binaria
df_clean = df_clean[df_clean['Target'] != 'Enrolled']

#3. Mappatura: Dropout = 1 (Classe da prevedere), Graduate = 0
df_clean['Target'] = df_clean['Target'].map({'Dropout': 1, 'Graduate': 0})

# separazione tra feature indipendenti(X) e target(y)
X = df_clean.drop('Target', axis=1)
y = df_clean['Target']

#4. Normalizzazione delle feature numeriche
# Questo passaggio è fondamentale per far convergere velocemente algoritmi basati sul calcolo
# delle distanze o del gradiente (come la Logistic Regression).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  #calcolo parametri e trasforma dati
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns) # DataFrame per mantenere i nomi delle colonne

# nuove dimensioni e percentuali delle classi per calcolare il grado di sbilanciamento
print(f"Dimensioni del dataset dopo il preprocessing: {X_scaled_df.shape}")
print(f"Distribuzione finale del Target:\n{y.value_counts(normalize=True).round(2)}")

# Fase 4: Applicazione modelli di classificazione

I dati vengono suddivisi in Training Set (80%) e Test Set (20%).
Vengono addestrati 3 diversi algoritmi per consentire un confronto prestazionale:
1. **Logistic Regression:** Modello baseline lineare.
2. **Random Forest:** Modello d'insieme (ensemble) robusto e ottimo per dati tabulari.
3. **Gradient Boosting:** Modello avanzato ad alte prestazioni.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# split stratificato per mantenere le proporzioni delle classi
# Il parametro 'stratify=y' garantisce che la proporzione tra Dropout e Graduate rimanga identica sia nel Train che nel Test set.
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.2, random_state=42, stratify=y)

# inizializzazione dei modelli
# Imposto random_state=42 per far sì che i risultati siano riproducibili ad ogni esecuzione.
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators = 100,random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# dizionario per immagazzinare i modelli addestrati
trained_models = {}

print("Addestramento in corso...")
for name, model in models.items():
    model.fit(X_train, y_train)     #addestramento
    trained_models[name] = model
    print(f"{name} addestrato con successo.")

# Fase 5: Valutazione e confronto modelli

Trattandosi di previsione di abbandono, l'accuratezza non è sufficiente. E' fondamentale osservare:
* **Recall (Sensibilità):** Quanti "Dropout" reali siamo riusciti ad individuare? (Evitare i falsi negativi è vitale per la segretaria universitaria).
* **F1-Score:** Media armonica tra Precision e Recall.

Verranno stampate le metriche e confrontate visivamente.

In [ ]:
# metriche per la valutazione dei modelli
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = {}

# Ciclo sui modelli addestrati per effettuare le predizioni sul Test Set
for name, model in trained_models.items():
    y_pred = model.predict(X_test)      # prevede chi abbandona e chi no
    
    acc = accuracy_score(y_test, y_pred)        # percentuale totale di predizioni esatte
    prec = precision_score(y_test, y_pred)      # su tutti i "Dropout" predetti quanti lo erano davvero
    rec = recall_score(y_test, y_pred)          # su tutti i Dropout REALI quanti sono stati indivduati
    f1 = f1_score(y_test, y_pred)               # media armonica tra Precision e Recall

    results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
    }

df_results = pd.DataFrame(results).T

#visualizzazione dei risultati
print("=== MATRICE DEI RISULTATI ===")
print(df_results.round(3))

#grafico a barre per confrontare Recall e F1-Score
df_results[['Recall', 'F1-Score']].plot(kind='bar', figsize=(10,5), colormap= 'Set2')
plt.title('Confronto Modelli: Recall vs F1-Score')
plt.ylabel('Punteggio')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(loc = 'lower right')
plt.show()

In [ ]:
# analisi dell'importanza delle feature (Feature Importance) con Random Forest
rf_model = trained_models['Random Forest']
importances = rf_model.feature_importances_

# DataFrame per ordinare le feature dalla più alla meno importante
feat_imp_df = pd.DataFrame({
    'Feature': X.columns,
    'Importanza': importances
}).sort_values(by='Importanza', ascending=False)

# grafico a barre orizzontali per le prime 10 feature
plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df.head(10), x='Importanza', y='Feature', hue='Feature', palette='magma', legend=False)
plt.title('Top 10 Fattori che influenzano l\'abbandono (Random Forest)')
plt.xlabel('Importanza Relativa')
plt.ylabel('Variabile (Feature)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

print("Come si nota dal grafico, i fattori accademici (es. voti e unità curriculari approvate nel 2° semestre) hanno un impatto " \
    "molto più forte rispetto ai dati anagrafici o macroeconomici.")

# Fase 6: Considerazioni finali

Analisi dei risultati ottenuti dal processo di Data Mining:
* Rimozione degli "Enrolled": ha permesso di creare un classificatore netto sui veri fattori di successo o fallimento.
* I modelli ad albero (Random Forest e Gradient Boosting) tendono a gestire meglio la complessità dei dati tabulari rispetto ai modelli lineari puri.
* Si è data priorità alla metrica **Recall** per minimizzare il rischio di ignorare studenti realmente in difficoltà.

In [ ]:
print(""" 
[CONSIDERAZIONI FINALI]
- L'esclusione degli studenti "Enrolled" ha reso il task di classificazione molto più robusto.
- I modelli ensemble (Random Forest e Gradient Boosting) hanno dimostrato ottime capacità di catturare i pattern legati al rischio accademico.
- Nelle metriche, la Recall ha un peso specifico elevato: dal punto di vista accademico/business, è preferibile avere un 'Falso Positivo' (intervenire su uno studente non a rischio) piuttosto che un 'Falso Negativo' (ignorare uno studente che finirà per abbandonare).
""")

# Fase 7: Prospettive di miglioramento

Possibili sviluppi futuri per espandere il progetto:
* **Interpretabilità (Explainable AI):** Implementare tecniche come SHAP per capire l'impatto delle singole feature (es. "Quanto pesano le assenze rispetto al voto di diploma?").
* **Tuning degli Iperparametri:** Utilizzare GridSearch o RandomSearch per ottimizzare ulteriormente il Gradient Boosting.
* **Gestione dello sbilanciamento:** Sperimentare tecniche come SMOTE (Synthetic Minority Over-sampling Technique) sul training set per aiutare il modello a riconoscere ancor meglio la classe minoritaria dei Dropout.

In [ ]:
print("""
[PROSPETTIVE FUTURE]
1. Integrazione della libreria SHAP (SHapley Additive exPlanations) per rendere il modello interpretabile ("White-box").
2. Hyperparameter Tuning approfondito tramite GridSearchCV.
3. Sperimentazione con librerie di Auto-ML o implementazione di reti neurali base.
4. Applicazione di tecniche di Oversampling (es. SMOTE) per valutare incrementi sulla metrica Recall.
""")

---